# RidgeVisionNet -- Part 1 of 3 (v2): Setup + Baseline Comparison

**This is a fresh re-run**, not a continuation of the original Part 1. It
includes two corrections discovered after the original run:

1. A fix for a bug where `tf.keras.applications.EfficientNetB0`'s built-in
   `Rescaling(1/255)` layer was double-scaling already-[0,1]-normalized input,
   crushing signal and causing training collapse -- this affected
   `efficientnet_b0_plain` here, and (separately, in Part 2) the
   `single_branch_texture` and `no_orientation_field` ablations, and possibly
   suppressed RidgeVisionNet's own accuracy since its appearance branch is
   also EfficientNetB0.
2. Per-baseline learning-rate overrides for MobileNetV2 and ResNet50
   (confirmed via an independent LR-search experiment to need a lower LR,
   unrelated to the EfficientNet bug above).

Run top to bottom, then **Save Version -> Save & Run All (Commit)**, then
create a Kaggle Dataset from the output named e.g. `ridgevisionnet-v2-part1-output`.


In [1]:
# !pip install -q scikit-image
import os, gc, json, time
from pathlib import Path

import cv2
import numpy as np
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from scipy import stats
from scipy.optimize import minimize_scalar

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))


TensorFlow: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [2]:
# =========================
# OFFLINE / NO-INTERNET RESILIENCE FOR IMAGENET WEIGHTS
# =========================
# Kaggle sessions default to "Internet: Off". Every backbone in this notebook
# (RidgeVisionNet + 6 of the 10 baselines) loads ImageNet-pretrained weights,
# which needs to download a .h5 file from storage.googleapis.com the first
# time it's used. If you saw an error like:
#   URLError: <urlopen error [Errno -3] Temporary failure in name resolution>
# it means Internet is Off for this session.
#
# FIX (do this first, it's the real fix): right sidebar -> Settings ->
# Internet -> toggle ON -> Save, then re-run. This changes nothing about the
# methodology -- it only lets the same published ImageNet weights download.
#
# The helpers below are a safety net for cases where you can't turn Internet
# on (e.g. a no-internet competition): they (1) detect the problem early with
# a clear message instead of a deep stack trace, (2) reuse any already-
# downloaded weights from a locally attached Kaggle dataset if one is
# present, and (3) as a last resort let backbone-building fall back to
# random-initialized weights (with a loud warning) so the notebook can still
# run to completion rather than crash -- note this last case means that
# backbone is no longer using transfer learning, which should be reported as
# a deviation if it happens for your real (non-QUICK_RUN) results.

import socket
import shutil


def internet_available(host="storage.googleapis.com", port=443, timeout=3):
    try:
        socket.getaddrinfo(host, port)
        return True
    except OSError:
        return False


HAS_INTERNET = internet_available()
print("Internet reachable:", HAS_INTERNET)


def stage_offline_imagenet_weights():
    """If a Kaggle dataset containing pre-downloaded Keras ImageNet weight
    files is attached (search Kaggle Datasets for 'keras pretrained models'
    or similar), copy any .h5 files found under /kaggle/input into
    ~/.keras/models/ so tf.keras.applications finds them in its local cache
    and skips the network call entirely. Safe to call even if nothing is
    found (returns 0)."""
    cache_dir = Path.home() / ".keras" / "models"
    cache_dir.mkdir(parents=True, exist_ok=True)
    found = 0
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for h5_path in input_root.rglob("*.h5"):
            target = cache_dir / h5_path.name
            if not target.exists():
                shutil.copy(h5_path, target)
                found += 1
    if found:
        print(f"Staged {found} local weight file(s) into {cache_dir}.")
    else:
        print("No local ImageNet weight files found under /kaggle/input.")
    return found


if not HAS_INTERNET:
    staged = stage_offline_imagenet_weights()
    if not staged:
        print()
        print("ACTION SUGGESTED: no internet reachable AND no local weights dataset found.")
        print("Go to Settings -> Internet -> ON (right sidebar), Save, and re-run this cell.")
        print("If your competition disallows internet, attach a Kaggle dataset that hosts the")
        print("*_notop.h5 files for the backbones used here, then re-run this cell.")


def load_backbone(backbone_cls, weights="imagenet", **kwargs):
    """Build a tf.keras.applications backbone, falling back to random-init
    weights (with a loud, impossible-to-miss warning) if the requested
    weights can't be obtained. Keeps the notebook runnable end-to-end even
    when Internet is Off and no offline weights dataset is attached; does
    NOT fix the underlying cause -- see the cell above."""
    try:
        return backbone_cls(weights=weights, **kwargs)
    except Exception as e:
        if weights is None:
            raise
        print(f"WARNING: could not load '{weights}' weights for {backbone_cls.__name__} ({type(e).__name__}: {e}).")
        print("Falling back to random initialization (weights=None) so the run can continue.")
        print("This backbone will NOT benefit from ImageNet transfer learning until the")
        print("internet/offline-weights issue above is resolved and this cell is re-run.")
        return backbone_cls(weights=None, **kwargs)


Internet reachable: True


In [3]:
# =========================
# CONFIG
# =========================
OUTPUT_DIR = Path('/kaggle/working')
RESULTS_DIR = OUTPUT_DIR / 'ridgevisionnet_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CLASS_LABELS = ['A+', 'A-', 'AB+', 'AB-', 'B+', 'B-', 'O+', 'O-']
LABEL_TO_INDEX = {label: i for i, label in enumerate(CLASS_LABELS)}
NUM_CLASSES = len(CLASS_LABELS)
SEED = 42

# --- Speed settings, changed this pass to fit a single Kaggle GPU session ---
# Mixed precision (fp16 compute, fp32 master weights) is a standard, purely
# computational optimization on GPUs with tensor cores (T4/P100/V100/A100) --
# it does not change what the model learns, only how fast the same forward/
# backward math runs, typically ~1.5-2x on these GPUs.
tf.keras.mixed_precision.set_global_policy('mixed_float16')

BATCH_SIZE = 32  # was 16; larger batches use the GPU more efficiently
IMG_SIZE = 224  # RidgeVisionNet / most baselines; InceptionV3 baseline overrides to 299

QUICK_RUN = False  # set False for the real run used in the paper
# EPOCHS_HEAD/EPOCHS_FINE are now upper bounds, not targets -- EarlyStopping
# (added this pass, see train_model) will stop well before these ceilings for
# any model that has converged or plateaued, which was most of the wasted
# time in the previous run (e.g. mobilenet_v2 ran all 55 epochs stuck near
# chance accuracy). Lowering the ceiling itself additionally caps worst-case
# runtime for a model that never triggers early stopping.
EPOCHS_HEAD = 3 if QUICK_RUN else 8
EPOCHS_FINE = 5 if QUICK_RUN else 30  # NOT lowered further than this: your own log showed resnet50/densenet121 flat until fine-tune epoch ~14, then climbing through epoch 40 -- a lower ceiling would cut those off mid-breakthrough and understate their real accuracy. EarlyStopping (patience=4) does the actual time-saving for models that plateau, not this ceiling.
# N_FOLDS reduced from 5 to 3: 3-fold stratified CV is still a standard,
# citable, defensible choice (commonly used exactly for compute-constrained
# settings) -- report it as "3-fold" rather than "5-fold" in the paper's
# Experimental Setup section rather than silently changing the number.
N_FOLDS = 2 if QUICK_RUN else 3
N_PERMUTATIONS = 200 if QUICK_RUN else 2000
MC_DROPOUT_SAMPLES = 10 if QUICK_RUN else 30

np.random.seed(SEED)
tf.random.set_seed(SEED)
print('QUICK_RUN =', QUICK_RUN)
print('Mixed precision policy:', tf.keras.mixed_precision.global_policy())


QUICK_RUN = False
Mixed precision policy: <DTypePolicy "mixed_float16">


In [4]:
# =========================
# DATASET AUTO-DETECTION (same convention as the v1 notebook)
# =========================

def find_dataset_dir():
    search_roots = [Path('/kaggle/input'), Path('/kaggle/working')]
    candidates = []
    for root in search_roots:
        if not root.exists():
            continue
        for path in root.rglob('*'):
            if not path.is_dir():
                continue
            class_folder_count = sum((path / label).is_dir() for label in CLASS_LABELS)
            if class_folder_count >= 6:
                candidates.append((class_folder_count, path))
    if not candidates:
        raise FileNotFoundError(
            'Dataset not found. Attach the fingerprint blood group dataset to Kaggle, '
            'or set DATASET_DIR manually below.'
        )
    candidates.sort(key=lambda item: item[0], reverse=True)
    print('Auto-detected dataset folder:', candidates[0][1])
    return candidates[0][1]

DATASET_DIR = find_dataset_dir()

all_paths, all_labels = [], []
for label in CLASS_LABELS:
    files = sorted((DATASET_DIR / label).glob('*'))
    all_paths.extend(files)
    all_labels.extend([LABEL_TO_INDEX[label]] * len(files))
all_labels = np.array(all_labels)
print(f'Total images: {len(all_paths)}')
for label in CLASS_LABELS:
    print(f'  {label}: {(all_labels == LABEL_TO_INDEX[label]).sum()}')


Auto-detected dataset folder: /kaggle/input/datasets/sravani2006/fingerprint-blood-group-classification-dataset/datasets
Total images: 5837
  A+: 402
  A-: 1009
  AB+: 708
  AB-: 761
  B+: 652
  B-: 741
  O+: 852
  O-: 712


In [5]:
# =========================
# IMAGE LOADING
# =========================

def load_rgb(path, img_size):
    img = cv2.imread(str(path))
    if img is None:
        return np.zeros((img_size, img_size, 3), dtype=np.float32)
    img = cv2.resize(img, (img_size, img_size))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img.astype(np.float32) / 255.0


class FingerprintSequence(tf.keras.utils.Sequence):
    """Simple image-only generator (RidgeVisionNet computes its own ridge
    orientation field on-device from the image, so no separate texture-vector
    input is needed here, unlike the v1 dual-input generator)."""

    def __init__(self, paths, labels, img_size, batch_size=BATCH_SIZE, augment=False, shuffle=True, **kwargs):
        super().__init__(**kwargs)  # required by Keras 3's PyDataset base (tf.keras.utils.Sequence is now an alias for it)
        self.paths = paths
        self.labels = labels
        self.img_size = img_size
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle
        self.indices = np.arange(len(paths))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.paths) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def _augment(self, img):
        if np.random.rand() < 0.5:
            angle = np.random.uniform(-10, 10)
            m = cv2.getRotationMatrix2D((self.img_size / 2, self.img_size / 2), angle, 1.0)
            img = cv2.warpAffine(img, m, (self.img_size, self.img_size), borderMode=cv2.BORDER_REFLECT)
        if np.random.rand() < 0.5:
            img = np.clip(img * np.random.uniform(0.85, 1.15) + np.random.uniform(-0.05, 0.05), 0, 1)
        if np.random.rand() < 0.3:
            img = np.clip(img + np.random.normal(0, 0.02, img.shape), 0, 1)
        return img.astype(np.float32)

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        images = np.zeros((len(batch_idx), self.img_size, self.img_size, 3), dtype=np.float32)
        labels = np.zeros(len(batch_idx), dtype=np.int32)
        for i, bi in enumerate(batch_idx):
            img = load_rgb(self.paths[bi], self.img_size)
            if self.augment:
                img = self._augment(img)
            images[i] = img
            labels[i] = self.labels[bi]
        return images, labels


## RidgeVisionNet components

Ridge orientation field (deterministic), ROAM (orientation-gated attention),
Adaptive Gated Fusion, and the full model builder -- identical logic to
`backend/ml/models/{ridge_orientation,roam,ridgevision_net}.py` in the
repository, inlined here so this notebook is self-contained on Kaggle.

In [6]:
# =========================
# Ridge Orientation Field (deterministic, no trainable params)
# =========================
class RidgeOrientationField(tf.keras.layers.Layer):
    def __init__(self, block_size=8, **kwargs):
        super().__init__(**kwargs)
        self.block_size = block_size
        self.sobel_x = tf.constant([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=tf.float32)[:, :, None, None]
        self.sobel_y = tf.constant([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=tf.float32)[:, :, None, None]

    def call(self, inputs):
        gx = tf.nn.conv2d(inputs, self.sobel_x, strides=1, padding='SAME')
        gy = tf.nn.conv2d(inputs, self.sobel_y, strides=1, padding='SAME')
        gxx, gyy, gxy = gx * gx, gy * gy, gx * gy
        pool = lambda t: tf.nn.avg_pool2d(t, ksize=self.block_size, strides=self.block_size, padding='VALID')
        vxx, vyy, vxy = pool(gxx), pool(gyy), pool(gxy)
        numerator = 2.0 * vxy
        denominator = vxx - vyy
        theta2 = tf.atan2(numerator, denominator)
        cos2, sin2 = tf.cos(theta2), tf.sin(theta2)
        energy = tf.sqrt(numerator**2 + denominator**2)
        coherence = tf.clip_by_value(energy / (vxx + vyy + 1e-6), 0.0, 1.0)
        return tf.concat([cos2, sin2, coherence], axis=-1)

    def get_config(self):
        config = super().get_config(); config.update({'block_size': self.block_size}); return config


class ROAM(tf.keras.layers.Layer):
    """Ridge Orientation Attention Module."""
    def __init__(self, reduction=4, use_channel_gate=True, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction
        # FIX: use_channel_gate must be a real constructor arg that build()/call()
        # branch on. Previously callers tried to monkeypatch `roam.channel_excite`
        # with a Lambda *before* the first call -- but build() runs on that first
        # call and unconditionally overwrote it with a trainable Dense layer, so
        # the "no_roam_channel_gate" ablation silently trained an unmodified model.
        self.use_channel_gate = use_channel_gate

    def build(self, input_shapes):
        feature_shape, _ = input_shapes
        channels = int(feature_shape[-1])
        reduced = max(channels // self.reduction, 8)
        self.orientation_proj = tf.keras.layers.Conv2D(reduced, 3, padding='same', activation='relu')
        self.spatial_gate = tf.keras.layers.Conv2D(1, 1, padding='same', activation='sigmoid')
        if self.use_channel_gate:
            self.channel_squeeze = tf.keras.layers.Dense(reduced, activation='relu')
            self.channel_excite = tf.keras.layers.Dense(channels, activation='sigmoid')
        self.resize_target = (int(feature_shape[1]), int(feature_shape[2]))

    def call(self, inputs):
        features, orientation_field = inputs
        orientation_resized = tf.image.resize(orientation_field, self.resize_target, method='bilinear')
        spatial_attention = self.spatial_gate(self.orientation_proj(orientation_resized))
        if self.use_channel_gate:
            channel_stats = tf.reduce_mean(features, axis=[1, 2])
            channel_attention = self.channel_excite(self.channel_squeeze(channel_stats))[:, None, None, :]
        else:
            channel_attention = 1.0  # true no-op: skips the gate entirely instead of shape-mismatched ones_like
        attended = features * spatial_attention * channel_attention
        return attended, spatial_attention

    def get_config(self):
        config = super().get_config()
        config.update({'reduction': self.reduction, 'use_channel_gate': self.use_channel_gate})
        return config


class AdaptiveGatedFusion(tf.keras.layers.Layer):
    def build(self, input_shapes):
        a_shape, b_shape = input_shapes
        dim = max(int(a_shape[-1]), int(b_shape[-1]))
        self.proj_a = tf.keras.layers.Dense(dim)
        self.proj_b = tf.keras.layers.Dense(dim)
        self.gate_dense = tf.keras.layers.Dense(dim, activation='sigmoid')

    def call(self, inputs):
        branch_a, branch_b = inputs
        a, b = self.proj_a(branch_a), self.proj_b(branch_b)
        gate = self.gate_dense(tf.concat([a, b], axis=-1))
        return gate * a + (1.0 - gate) * b, gate


In [7]:
# =========================
# RidgeVisionNet builder (+ ablation-variant builder)
# =========================

def build_ridgevision_net(img_size=IMG_SIZE, num_classes=NUM_CLASSES, dropout_rate=0.35,
                           trainable_backbone_layers=40, use_orientation_field=True,
                           use_channel_gate=True, fusion_mode='adaptive',
                           use_ridge_branch=True, use_appearance_branch=True, name='ridgevision_net'):
    assert use_ridge_branch or use_appearance_branch
    image_input = tf.keras.Input(shape=(img_size, img_size, 3), name='fingerprint_image')
    # FIX: tf.keras.applications.EfficientNetB0 has a built-in Rescaling(1/255)
    # layer expecting raw [0,255] pixels. Our shared pipeline already scales
    # images to [0,1] (Section 7.3), so without this correction EfficientNetB0
    # was dividing already-scaled pixels by 255 again -- crushing its input by
    # another factor of 255 and very likely causing the training collapses
    # observed in efficientnet_b0_plain, single_branch_texture, and
    # no_orientation_field. This undoes that scaling immediately before the
    # backbone; the grayscale/orientation-field path below is untouched since
    # it operates on the original [0,1] image_input, not this rescaled copy.
    efficientnet_input = tf.keras.layers.Rescaling(255.0, name='undo_pipeline_rescale_for_efficientnet')(image_input)
    backbone = load_backbone(tf.keras.applications.EfficientNetB0, weights='imagenet', include_top=False, input_tensor=efficientnet_input)
    for layer in backbone.layers:
        layer.trainable = False
    if trainable_backbone_layers > 0:
        for layer in backbone.layers[-trainable_backbone_layers:]:
            if not isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = True

    branches = []
    if use_ridge_branch:
        mid_features = backbone.get_layer('block6a_expand_activation').output
        if use_orientation_field:
            # Keras 3: raw tf.* ops cannot be applied directly to a KerasTensor
            # outside of a Layer; wrap in Lambda so it becomes a proper graph op.
            grayscale = tf.keras.layers.Lambda(
                lambda x: tf.image.rgb_to_grayscale(x), name='to_grayscale'
            )(image_input)
            orientation_field = RidgeOrientationField(block_size=8, dtype='float32')(grayscale)  # force float32: mixed precision would otherwise feed float16 into the hardcoded float32 Sobel kernels
        else:
            orientation_field = tf.keras.layers.Lambda(
                lambda x: tf.ones((tf.shape(x)[0], tf.shape(x)[1], tf.shape(x)[2], 3)),
                name='dummy_orientation_field'
            )(mid_features)
        roam = ROAM(use_channel_gate=use_channel_gate)
        attended_mid, spatial_attention = roam([mid_features, orientation_field])
        ridge_branch = tf.keras.layers.GlobalAveragePooling2D()(attended_mid)
        branches.append(ridge_branch)
    else:
        spatial_attention = None

    if use_appearance_branch:
        appearance_branch = tf.keras.layers.GlobalAveragePooling2D()(backbone.output)
        branches.append(appearance_branch)

    if len(branches) == 1:
        fused = branches[0]
    elif fusion_mode == 'adaptive':
        fused, _ = AdaptiveGatedFusion()(branches)
    elif fusion_mode == 'static_average':
        dim = 256
        projected = [tf.keras.layers.Dense(dim)(b) for b in branches]
        fused = tf.keras.layers.Average()(projected)
    elif fusion_mode == 'concat':
        fused = tf.keras.layers.Concatenate()(branches)
    else:
        raise ValueError(fusion_mode)

    hidden = tf.keras.layers.Dense(256, activation='relu')(fused)
    hidden = tf.keras.layers.Dropout(dropout_rate, name='mc_dropout')(hidden)
    logits = tf.keras.layers.Dense(num_classes, name='logits', dtype='float32')(hidden)
    probs = tf.keras.layers.Softmax(name='blood_group', dtype='float32')(logits)

    outputs = {'blood_group': probs, 'logits': logits}
    if spatial_attention is not None:
        outputs['attention_map'] = spatial_attention
    return tf.keras.Model(inputs=image_input, outputs=outputs, name=name)


ABLATION_GRID = [
    dict(name='full'),
    dict(name='no_orientation_field', use_orientation_field=False),
    dict(name='no_roam_channel_gate', use_channel_gate=False),
    dict(name='static_fusion', fusion_mode='static_average'),
    dict(name='concat_fusion', fusion_mode='concat'),
    dict(name='single_branch_texture', use_ridge_branch=False),
    dict(name='single_branch_ridge', use_appearance_branch=False),
    dict(name='no_finetune', trainable_backbone_layers=0),
]
print(f'{len(ABLATION_GRID)} ablation variants configured.')


8 ablation variants configured.


In [8]:
# =========================
# Baseline builders (10 methods for Table 7.1)
# =========================
CNN_BASELINES = {
    'mobilenet_v2': ('MobileNetV2', 224),
    'resnet50': ('ResNet50', 224),
    'densenet121': ('DenseNet121', 224),
    'inception_v3': ('InceptionV3', 299),
    'efficientnet_b0_plain': ('EfficientNetB0', 224),
    'convnext_tiny': ('ConvNeXtTiny', 224),
}

def build_cnn_baseline(key, num_classes=NUM_CLASSES, trainable_layers=40):
    backbone_cls_name, img_size = CNN_BASELINES[key]
    backbone_cls = getattr(tf.keras.applications, backbone_cls_name)
    image_input = tf.keras.Input(shape=(img_size, img_size, 3))
    # FIX: EfficientNetB0 (unlike the other backbones in CNN_BASELINES) has a
    # built-in Rescaling(1/255) layer expecting raw [0,255] pixels; our shared
    # pipeline already scales to [0,1], so without this correction it was
    # dividing by 255 twice, crushing the input and causing training collapse
    # regardless of learning rate.
    if backbone_cls_name == 'EfficientNetB0':
        backbone_input = tf.keras.layers.Rescaling(255.0, name='undo_pipeline_rescale_for_efficientnet')(image_input)
    else:
        backbone_input = image_input
    backbone = load_backbone(backbone_cls, weights='imagenet', include_top=False, input_tensor=backbone_input)
    for layer in backbone.layers:
        layer.trainable = False
    for layer in backbone.layers[-trainable_layers:]:
        if not isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = True
    pooled = tf.keras.layers.GlobalAveragePooling2D()(backbone.output)
    hidden = tf.keras.layers.Dense(256, activation='relu')(pooled)
    hidden = tf.keras.layers.Dropout(0.35)(hidden)
    output = tf.keras.layers.Dense(num_classes, activation='softmax', name='blood_group', dtype='float32')(hidden)
    return tf.keras.Model(inputs=image_input, outputs=output, name=f'baseline_{key}'), img_size


def build_plain_cnn_from_scratch(num_classes=NUM_CLASSES, img_size=224):
    return tf.keras.Sequential([
        tf.keras.Input(shape=(img_size, img_size, 3)),
        tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(256, 3, activation='relu', padding='same'),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(num_classes, activation='softmax', name='blood_group', dtype='float32'),
    ], name='plain_cnn_from_scratch'), img_size


def extract_handcrafted_features(paths, img_size=128):
    """Reuses the same LBP/GLCM/ridge descriptor family as v1's texture.py,
    for the SVM / Random Forest classical-ML baselines."""
    from skimage.feature import local_binary_pattern, graycomatrix, graycoprops
    from skimage.measure import shannon_entropy
    feats = []
    for p in paths:
        gray = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        gray = cv2.resize(gray, (img_size, img_size)) if gray is not None else np.zeros((img_size, img_size), np.uint8)
        lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')
        hist, _ = np.histogram(lbp, bins=10, range=(0, 10), density=True)
        glcm = graycomatrix(gray, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
        glcm_feats = [graycoprops(glcm, p)[0, 0] for p in ('contrast', 'homogeneity', 'energy', 'correlation')]
        edges = cv2.Canny(gray, 50, 150)
        ridge_density = edges.mean() / 255.0
        entropy = shannon_entropy(gray)
        feats.append(list(hist) + glcm_feats + [ridge_density, entropy, gray.mean(), gray.std()])
    return np.array(feats)


def build_classical_ml_baselines():
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.svm import SVC
    return {
        'svm_rbf': SVC(kernel='rbf', C=10.0, gamma='scale', probability=True, class_weight='balanced'),
        'random_forest': RandomForestClassifier(n_estimators=400, class_weight='balanced', random_state=SEED),
    }

print('Baselines configured:', list(CNN_BASELINES) + ['plain_cnn_from_scratch', 'svm_rbf', 'random_forest'])


Baselines configured: ['mobilenet_v2', 'resnet50', 'densenet121', 'inception_v3', 'efficientnet_b0_plain', 'convnext_tiny', 'plain_cnn_from_scratch', 'svm_rbf', 'random_forest']


## Training utility (shared recipe across RidgeVisionNet, baselines, and ablations)

In [9]:
def train_model(model, train_seq, val_seq, epochs_head=EPOCHS_HEAD, epochs_fine=EPOCHS_FINE,
                 class_weight=None, is_dict_output=True):
    output_name = 'blood_group' if is_dict_output else None
    loss = {output_name: 'sparse_categorical_crossentropy'} if is_dict_output else 'sparse_categorical_crossentropy'
    metrics = {output_name: 'accuracy'} if is_dict_output else ['accuracy']
    monitor = 'val_loss'  # verified key name for both dict-output and plain models

    # EarlyStopping cuts epochs once val loss stops improving (patience=6) and
    # restores the best-seen weights. This does not change the training
    # recipe's intent (same optimizer/LR schedule) -- it just stops paying for
    # epochs that were already flat/plateaued or had started overfitting,
    # which is where most of the wasted time in a fixed 15+40-epoch schedule
    # goes once a model has converged (or gotten stuck, as with mobilenet_v2).
    early_stop = tf.keras.callbacks.EarlyStopping(monitor=monitor, patience=7, restore_best_weights=True)
    # patience=7 (raised from 4): the real run showed resnet50-style architectures can plateau
    # 10+ epochs before a late breakthrough. patience=4 was cutting that breakthrough off before
    # it happened -- correctness matters more than shaving GPU time here.

    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=loss, metrics=metrics)
    model.fit(train_seq, validation_data=val_seq, epochs=epochs_head, class_weight=class_weight,
              callbacks=[early_stop], verbose=2)

    # unfreeze already-configured trainable layers and fine-tune at low LR
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss=loss, metrics=metrics)
    model.fit(train_seq, validation_data=val_seq, epochs=epochs_fine, class_weight=class_weight,
              callbacks=[early_stop], verbose=2)
    return model


def evaluate_model(model, test_seq, labels, is_dict_output=True):
    preds = model.predict(test_seq, verbose=0)
    probs = preds['blood_group'] if is_dict_output else preds
    y_pred = probs.argmax(axis=1)
    acc = accuracy_score(labels, y_pred)
    return acc, probs, y_pred


# =========================
# TUNED TRAINING HELPER (adds configurable head_lr/finetune_lr)
# =========================
# train_model() above hardcodes Adam(1e-3) / Adam(1e-5). MobileNetV2 and
# ResNet50 were independently confirmed (LR-search experiment) to collapse to
# near-chance accuracy at that default head LR but train normally at a lower
# one -- unrelated to the EfficientNetB0 rescaling bug fixed elsewhere in this
# notebook. This variant exposes the LR so the baseline-comparison loop can
# use a per-method override for those two specifically, while every other
# method keeps the original, unmodified recipe.
def train_model_tuned(model, train_seq, val_seq, epochs_head=EPOCHS_HEAD, epochs_fine=EPOCHS_FINE,
                       head_lr=1e-3, finetune_lr=1e-5, class_weight=None, is_dict_output=True, patience=7):
    output_name = 'blood_group' if is_dict_output else None
    loss = {output_name: 'sparse_categorical_crossentropy'} if is_dict_output else 'sparse_categorical_crossentropy'
    metrics = {output_name: 'accuracy'} if is_dict_output else ['accuracy']
    early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True)

    model.compile(optimizer=tf.keras.optimizers.Adam(head_lr), loss=loss, metrics=metrics)
    model.fit(train_seq, validation_data=val_seq, epochs=epochs_head, class_weight=class_weight,
              callbacks=[early_stop], verbose=2)

    model.compile(optimizer=tf.keras.optimizers.Adam(finetune_lr), loss=loss, metrics=metrics)
    model.fit(train_seq, validation_data=val_seq, epochs=epochs_fine, class_weight=class_weight,
              callbacks=[early_stop], verbose=2)
    return model

# Per-baseline LR overrides confirmed via independent LR-search experiment.
# Methods not listed here use the original default (1e-3, 1e-5) recipe.
BASELINE_LR_OVERRIDES = {
    'mobilenet_v2': (3e-4, 3e-6),
    'resnet50': (3e-5, 3e-7),
}


## 1. RidgeVisionNet vs. 10 baselines, 5-fold stratified CV + Wilcoxon test

This is the experiment behind Table 7.1. Each fold trains RidgeVisionNet and
every baseline from the same split, so the paired Wilcoxon test is valid.

In [10]:
all_paths_arr = np.array(all_paths, dtype=object)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

CHECKPOINT_PATH = RESULTS_DIR / 'baseline_comparison_fold_results.json'
MODEL_NAMES = ['ridgevision_net'] + list(CNN_BASELINES) + ['plain_cnn_from_scratch', 'svm_rbf', 'random_forest']

# RESUME SUPPORT: if this cell was interrupted (Kaggle's session time limit,
# a disconnect, etc.), reload whatever was already checkpointed and skip
# (fold, model) combinations already completed, instead of restarting from
# scratch. Every fold/model result is written to disk immediately after it
# finishes (not just once at the very end), so at most one model's worth of
# a single fold is ever at risk if the session is interrupted mid-run.
if CHECKPOINT_PATH.exists():
    with open(CHECKPOINT_PATH) as f:
        fold_results = json.load(f)
    print(f"Resuming from checkpoint: {CHECKPOINT_PATH}")
    for name in MODEL_NAMES:
        fold_results.setdefault(name, [])
else:
    fold_results = {name: [] for name in MODEL_NAMES}

def save_checkpoint():
    with open(CHECKPOINT_PATH, 'w') as f:
        json.dump(fold_results, f, indent=2)

for fold, (train_idx, test_idx) in enumerate(skf.split(all_paths_arr, all_labels)):
    # Skip this fold entirely if every model already has a result recorded for it.
    if all(len(fold_results[name]) > fold for name in MODEL_NAMES):
        print(f'Fold {fold + 1}/{N_FOLDS} already completed in a previous run -- skipping.')
        continue

    print(f'\n=== Fold {fold + 1}/{N_FOLDS} ===')
    train_idx, val_idx = train_test_split(train_idx, test_size=0.15, stratify=all_labels[train_idx], random_state=SEED)

    train_paths = all_paths_arr[train_idx]; train_labels = all_labels[train_idx]
    val_paths = all_paths_arr[val_idx]; val_labels = all_labels[val_idx]
    test_paths = all_paths_arr[test_idx]; test_labels = all_labels[test_idx]

    class_weight_vals = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=train_labels)
    class_weight = {i: w for i, w in enumerate(class_weight_vals)}

    # --- RidgeVisionNet ---
    if len(fold_results['ridgevision_net']) <= fold:
        train_seq = FingerprintSequence(train_paths, train_labels, IMG_SIZE, augment=True)
        val_seq = FingerprintSequence(val_paths, val_labels, IMG_SIZE, augment=False, shuffle=False)
        test_seq = FingerprintSequence(test_paths, test_labels, IMG_SIZE, augment=False, shuffle=False)

        model = build_ridgevision_net(name=f'ridgevision_net_fold{fold}')
        train_model(model, train_seq, val_seq, class_weight=class_weight)
        acc, probs, y_pred = evaluate_model(model, test_seq, test_labels)
        fold_results['ridgevision_net'].append(acc)
        print('RidgeVisionNet fold acc:', acc)
        save_checkpoint()
        del model; gc.collect(); tf.keras.backend.clear_session()
    else:
        print('ridgevision_net fold', fold, 'already done -- skipping.')

    # --- CNN baselines ---
    for key in CNN_BASELINES:
        if len(fold_results[key]) > fold:
            print(key, 'fold', fold, 'already done -- skipping.')
            continue
        _, img_size = CNN_BASELINES[key]
        tr_seq = FingerprintSequence(train_paths, train_labels, img_size, augment=True)
        va_seq = FingerprintSequence(val_paths, val_labels, img_size, augment=False, shuffle=False)
        te_seq = FingerprintSequence(test_paths, test_labels, img_size, augment=False, shuffle=False)
        model, _ = build_cnn_baseline(key)
        if key in BASELINE_LR_OVERRIDES:
            head_lr, finetune_lr = BASELINE_LR_OVERRIDES[key]
            train_model_tuned(model, tr_seq, va_seq, head_lr=head_lr, finetune_lr=finetune_lr,
                               class_weight=class_weight, is_dict_output=False)
        else:
            train_model(model, tr_seq, va_seq, class_weight=class_weight, is_dict_output=False)
        acc, _, _ = evaluate_model(model, te_seq, test_labels, is_dict_output=False)
        fold_results[key].append(acc)
        print(f'{key} fold acc:', acc)
        save_checkpoint()
        del model; gc.collect(); tf.keras.backend.clear_session()

    # --- plain CNN from scratch ---
    if len(fold_results['plain_cnn_from_scratch']) <= fold:
        tr_seq = FingerprintSequence(train_paths, train_labels, 224, augment=True)
        va_seq = FingerprintSequence(val_paths, val_labels, 224, augment=False, shuffle=False)
        te_seq = FingerprintSequence(test_paths, test_labels, 224, augment=False, shuffle=False)
        model, _ = build_plain_cnn_from_scratch()
        train_model(model, tr_seq, va_seq, class_weight=class_weight, is_dict_output=False)
        acc, _, _ = evaluate_model(model, te_seq, test_labels, is_dict_output=False)
        fold_results['plain_cnn_from_scratch'].append(acc)
        save_checkpoint()
        del model; gc.collect(); tf.keras.backend.clear_session()
    else:
        print('plain_cnn_from_scratch fold', fold, 'already done -- skipping.')

    # --- classical ML baselines on handcrafted features ---
    if len(fold_results['svm_rbf']) <= fold or len(fold_results['random_forest']) <= fold:
        X_train = extract_handcrafted_features(train_paths)
        X_test = extract_handcrafted_features(test_paths)
        for clf_name, clf in build_classical_ml_baselines().items():
            if len(fold_results[clf_name]) > fold:
                continue
            clf.fit(X_train, train_labels)
            acc = accuracy_score(test_labels, clf.predict(X_test))
            fold_results[clf_name].append(acc)
            print(f'{clf_name} fold acc:', acc)
        save_checkpoint()

print('\nAll folds complete. Results saved incrementally to', CHECKPOINT_PATH)



=== Fold 1/3 ===


I0000 00:00:1783846556.074673      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783846556.077652      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/8


2026-07-12 08:56:45.539068: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-12 08:56:45.696387: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-12 08:56:45.871546: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-12 08:56:46.038916: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1783846624.603707      70 device_compiler.h:196]

104/104 - 249s - 2s/step - loss: 1.3364 - val_loss: 0.6907
Epoch 2/8
104/104 - 18s - 176ms/step - loss: 0.6882 - val_loss: 0.5075
Epoch 3/8
104/104 - 19s - 180ms/step - loss: 0.5472 - val_loss: 0.5647
Epoch 4/8
104/104 - 18s - 169ms/step - loss: 0.4870 - val_loss: 0.4461
Epoch 5/8
104/104 - 18s - 173ms/step - loss: 0.4258 - val_loss: 0.4037
Epoch 6/8
104/104 - 19s - 183ms/step - loss: 0.3772 - val_loss: 0.3586
Epoch 7/8
104/104 - 17s - 167ms/step - loss: 0.3397 - val_loss: 0.3883
Epoch 8/8
104/104 - 18s - 170ms/step - loss: 0.3397 - val_loss: 0.3344
Epoch 1/30
104/104 - 130s - 1s/step - loss: 0.2765 - val_loss: 0.2818
Epoch 2/30
104/104 - 17s - 163ms/step - loss: 0.2259 - val_loss: 0.2725
Epoch 3/30
104/104 - 16s - 155ms/step - loss: 0.2144 - val_loss: 0.2644
Epoch 4/30
104/104 - 16s - 154ms/step - loss: 0.2017 - val_loss: 0.2642
Epoch 5/30
104/104 - 16s - 154ms/step - loss: 0.1839 - val_loss: 0.2641
Epoch 6/30
104/104 - 15s - 145ms/step - loss: 0.1814 - val_loss: 0.2629
Epoch 7/30
104

...

mobilenet_v2 fold acc: 0.9146529562982005
Epoch 1/8
104/104 - 44s - 423ms/step - accuracy: 0.1581 - loss: 2.0622 - val_accuracy: 0.2226 - val_loss: 1.8965
Epoch 2/8
104/104 - 16s - 158ms/step - accuracy: 0.3428 - loss: 1.6934 - val_accuracy: 0.5154 - val_loss: 1.3293
Epoch 3/8
104/104 - 16s - 154ms/step - accuracy: 0.4583 - loss: 1.3760 - val_accuracy: 0.5856 - val_loss: 1.0885
Epoch 4/8
104/104 - 17s - 162ms/step - accuracy: 0.5632 - loss: 1.1034 - val_accuracy: 0.5325 - val_loss: 1.1702
Epoch 5/8
104/104 - 16s - 155ms/step - accuracy: 0.6064 - loss: 0.9736 - val_accuracy: 0.6969 - val_loss: 0.7876
Epoch 6/8
104/104 - 16s - 151ms/step - accuracy: 0.6551 - loss: 0.8717 - val_accuracy: 0.6849 - val_loss: 0.8142
Epoch 7/8
104/104 - 16s - 156ms/step - accuracy: 0.6714 - loss: 0.8526 - val_accuracy: 0.6781 - val_loss: 0.8024
Epoch 8/8
104/104 - 16s - 155ms/step - accuracy: 0.6920 - loss: 0.7938 - val_accuracy: 0.7089 - val_loss: 0.7492
Epoch 1/30
104/104 - 42s - 402ms/step - accuracy: 0.76

2026-07-12 12:56:31.469636: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-12 12:56:31.604898: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


104/104 - 78s - 746ms/step - accuracy: 0.5333 - loss: 1.2295 - val_accuracy: 0.8116 - val_loss: 0.5521
Epoch 2/8
104/104 - 35s - 337ms/step - accuracy: 0.7669 - loss: 0.6018 - val_accuracy: 0.8014 - val_loss: 0.4870
Epoch 3/8
104/104 - 35s - 337ms/step - accuracy: 0.8207 - loss: 0.4755 - val_accuracy: 0.8527 - val_loss: 0.4144
Epoch 4/8
104/104 - 35s - 336ms/step - accuracy: 0.8371 - loss: 0.4013 - val_accuracy: 0.8151 - val_loss: 0.4501
Epoch 5/8
104/104 - 35s - 336ms/step - accuracy: 0.8507 - loss: 0.3822 - val_accuracy: 0.8442 - val_loss: 0.4100
Epoch 6/8
104/104 - 35s - 336ms/step - accuracy: 0.8643 - loss: 0.3537 - val_accuracy: 0.7962 - val_loss: 0.6302
Epoch 7/8
104/104 - 35s - 336ms/step - accuracy: 0.8755 - loss: 0.3071 - val_accuracy: 0.8579 - val_loss: 0.3466
Epoch 8/8
104/104 - 35s - 336ms/step - accuracy: 0.8987 - loss: 0.2743 - val_accuracy: 0.7928 - val_loss: 0.5679
Epoch 1/30
104/104 - 74s - 710ms/step - accuracy: 0.9353 - loss: 0.1931 - val_accuracy: 0.8801 - val_loss:

2026-07-12 13:13:16.802876: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-12 13:13:16.935722: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


convnext_tiny fold acc: 0.8956298200514139
Epoch 1/8
104/104 - 22s - 216ms/step - accuracy: 0.1557 - loss: 2.0317 - val_accuracy: 0.4349 - val_loss: 1.6461
Epoch 2/8
104/104 - 15s - 145ms/step - accuracy: 0.3981 - loss: 1.4639 - val_accuracy: 0.5086 - val_loss: 1.2828
Epoch 3/8
104/104 - 15s - 144ms/step - accuracy: 0.5142 - loss: 1.1940 - val_accuracy: 0.5531 - val_loss: 1.0848
Epoch 4/8
104/104 - 16s - 151ms/step - accuracy: 0.6236 - loss: 0.9607 - val_accuracy: 0.6182 - val_loss: 0.9238
Epoch 5/8
104/104 - 15s - 146ms/step - accuracy: 0.6644 - loss: 0.8465 - val_accuracy: 0.6935 - val_loss: 0.7513
Epoch 6/8
104/104 - 16s - 153ms/step - accuracy: 0.7134 - loss: 0.7065 - val_accuracy: 0.7414 - val_loss: 0.6822
Epoch 7/8
104/104 - 16s - 154ms/step - accuracy: 0.7300 - loss: 0.7009 - val_accuracy: 0.6644 - val_loss: 0.8756
Epoch 8/8
104/104 - 15s - 143ms/step - accuracy: 0.7412 - loss: 0.6549 - val_accuracy: 0.7534 - val_loss: 0.6118
Epoch 1/30
104/104 - 22s - 209ms/step - accuracy: 0.7

In [11]:
# Summarize mean/std and run paired Wilcoxon signed-rank test vs RidgeVisionNet
summary = {}
ridge_scores = np.array(fold_results['ridgevision_net'])
for name, scores in fold_results.items():
    scores = np.array(scores)
    entry = {'mean': float(scores.mean()), 'std': float(scores.std())}
    if name != 'ridgevision_net' and len(scores) == len(ridge_scores) and len(scores) > 1:
        try:
            stat, p = stats.wilcoxon(ridge_scores, scores)
            entry['wilcoxon_p_vs_ridgevisionnet'] = float(p)
        except ValueError as e:
            entry['wilcoxon_p_vs_ridgevisionnet'] = None
    summary[name] = entry

with open(RESULTS_DIR / 'baseline_comparison_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

for name, entry in sorted(summary.items(), key=lambda kv: -kv[1]['mean']):
    p = entry.get('wilcoxon_p_vs_ridgevisionnet')
    p_str = f"p={p:.4f}" if p is not None else ''
    print(f"{name:24s} {entry['mean']:.4f} +/- {entry['std']:.4f}  {p_str}")
print('\nThis table (mean/std/Wilcoxon p) is exactly what belongs in paper Table 1.')


mobilenet_v2             0.9107 +/- 0.0031  p=0.2500
ridgevision_net          0.8996 +/- 0.0064  
convnext_tiny            0.8991 +/- 0.0049  p=0.7500
efficientnet_b0_plain    0.8982 +/- 0.0088  p=1.0000
densenet121              0.8864 +/- 0.0026  p=0.2500
inception_v3             0.8504 +/- 0.0007  p=0.2500
plain_cnn_from_scratch   0.8223 +/- 0.0138  p=0.2500
resnet50                 0.7994 +/- 0.0066  p=0.2500
random_forest            0.3863 +/- 0.0031  p=0.2500
svm_rbf                  0.2465 +/- 0.0124  p=0.2500

This table (mean/std/Wilcoxon p) is exactly what belongs in paper Table 1.


---
### Part 1 (v2) done.

Save Version -> Save & Run All (Commit), then create a Kaggle Dataset from
the output. Attach it as Input to Part 2 (v2).
